In [ ]:
# AWS CLI ??
!pip install awscli

In [ ]:
# ????? ?? ???? ???? ? ??? ??? ??? ?? ??? ????.
!aws --version

In [ ]:
# AWS ?? ?? ??
!aws configure set aws_access_key_id {AWS_ACCESS_KEY}
!aws configure set aws_secret_access_key {AWS_SECRET_KEY}
!aws configure set default.region {REGION}

print(f"?{TEAM_NAME}? AWS ?? ??? ???????!")

***

??? ????

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import make_scorer, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.metrics import fbeta_score, precision_recall_curve, auc, recall_score, precision_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [ ]:
# train, test ??? ?? ????
df_train = pd.read_parquet("s3://epoch-mini-project-5th/Silver/real/user/train/train.parquet")
df_test = pd.read_parquet("s3://epoch-mini-project-5th/Silver/real/user/test/test.parquet")

## ??? ??

stratified split?? ?? ???? ???(??) ??? ??? ????? ???.

?? ??? 3% ??.

In [ ]:
# ID ?? ? ?? ?? ??
X = df_train.drop(columns=['ID', 'Bankrupt?'])
y = df_train['Bankrupt?']

# Stratified Holdout Split (20%? ?? ?????)
X_train_sub, X_valid_holdout, y_train_sub, y_valid_holdout = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train Sub Size: {X_train_sub.shape}, Holdout Size: {X_valid_holdout.shape}")
print(f"Train Sub Positive: {y_train_sub.sum()}, Holdout Positive: {y_valid_holdout.sum()}")

***

## ???

* Clipping?? ??? ??: ?? 1%, ?? 99% ???? ??? ?? 1%? 99%? ??? ??.
* Skewness: ??? ??? 1?? ??, ?? ?? 0 ??? ??? ?? ?? ??
* Fit vs Transform ??: ?? ????? ?/?? 1% ??? ??. ?? ???? ?? ???? ?? leakage??

In [ ]:
class FinancePreprocessor:
    def __init__(self, flag_cols=None, clip_lower=0.01, clip_upper=0.99, skew_threshold=1.0):
        self.flag_cols = flag_cols if flag_cols is not None else []
        self.clip_lower = clip_lower
        self.clip_upper = clip_upper
        self.skew_threshold = skew_threshold
        
        self.num_cols_ = None
        self.lower_bounds_ = None
        self.upper_bounds_ = None
        self.skewed_cols_ = None
        self.feature_order_ = None

    def fit(self, X: pd.DataFrame):
        X = X.copy()
        self.feature_order_ = X.columns.tolist()

        # flag ??? ?? ??? ??
        self.num_cols_ = [c for c in X.columns if c not in self.flag_cols]

        X_num = X[self.num_cols_].copy()

        # train_sub ?? clipping ?? ??
        self.lower_bounds_ = X_num.quantile(self.clip_lower)
        self.upper_bounds_ = X_num.quantile(self.clip_upper)

        # clipping ? ?? ??
        X_clipped = X_num.clip(
            lower=self.lower_bounds_,
            upper=self.upper_bounds_,
            axis=1
        )

        skewness = X_clipped.skew(numeric_only=True)

        # log1p? ?? ??? ?????, "clipping ??? ?? ?? 0 ??"? ??? ??
        candidate_cols = skewness[skewness > self.skew_threshold].index.tolist()
        self.skewed_cols_ = [c for c in candidate_cols if (X_clipped[c] >= 0).all()]

        return self

    def transform(self, X: pd.DataFrame):
        X = X.copy()

        # fit ? ? ?? ??? ??
        X = X[self.feature_order_].copy()

        # ?? ??? ??? ??
        X_num = X[self.num_cols_].copy()

        # train_sub?? ?? ???? clip
        X_num = X_num.clip(
            lower=self.lower_bounds_,
            upper=self.upper_bounds_,
            axis=1
        )

        # ?? ? ?? ??? log1p
        if len(self.skewed_cols_) > 0:
            X_num[self.skewed_cols_] = np.log1p(X_num[self.skewed_cols_])

        # ??? ??
        X_out = X.copy()
        X_out[self.num_cols_] = X_num

        return X_out

    def fit_transform(self, X: pd.DataFrame):
        return self.fit(X).transform(X)

In [ ]:
flag_cols = ['Liability-Assets Flag', 'Net Income Flag']

preprocessor = FinancePreprocessor(
    flag_cols=flag_cols,
    clip_lower=0.01,
    clip_upper=0.99,
    skew_threshold=1.0)

X_train_sub_prep = preprocessor.fit_transform(X_train_sub)
X_valid_holdout_prep = preprocessor.transform(X_valid_holdout)

print("?? shape:", X_train_sub.shape, X_valid_holdout.shape)
print("??? ? shape:", X_train_sub_prep.shape, X_valid_holdout_prep.shape)
print("log ?? ?? ?:", len(preprocessor.skewed_cols_))
print("?? log ?? ??:", preprocessor.skewed_cols_[:10])

## ?? ?? ??

logistic, RandomForest, XGBoost, LightGBM? ??? ??? ?? ??

* StrarifiedKFold: ???? 5? ???(Fold)? ?? ?, 4?? ???? 1?? ??
* OOF(Out-Of-Fold): ??? ??? ? ?? ??? ????? ?? ????? ???. ??? ?? ??? ??? ?? "? ?? ? ? ?? ???? ?" ?? ?? ??
* PR curve? ??: F2-Score? ??? ?? threshold? ???. ??? ??? 0.5? ??? ????? ???, ?? ??? ?? ??? ??? ??? ?? ? ?? ??. ??? 0.1?? 0.2?? ?? ????? ??? ??? ??? 1? ????? ??? ???(threshold)? ???? ??. F1? ?? F2? ????, "??? ????? ??(Precision ??)????, ?? ??? ??? ?(Recall ??)? ???" ???

In [ ]:
# =========================
# OOF ?? ?? ?? ??
# =========================
def evaluate_models_oof(X, y, random_state=42):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

    # ?? train_sub ?? imbalance ratio
    # RandomForest? Logistic ???? balanced? ??? ??
    # XGBoost? LightGBM ???? 1? ???? ???? 32? ?? ???? ????? ??
    pos = y.sum()
    neg = len(y) - pos
    scale_pos_weight = neg / pos

    models = {
        'Logistic': Pipeline([
            ('scaler', StandardScaler()),
            ('model', LogisticRegression(
                max_iter=3000,
                class_weight='balanced',
                random_state=random_state
            ))
        ]),
        'RandomForest': RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            class_weight='balanced',
            n_jobs=-1,
            random_state=random_state
        ),
        'XGBoost': XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=4,
            min_child_weight=1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            scale_pos_weight=scale_pos_weight,
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=random_state,
            tree_method='hist',
            n_jobs=-1
        ),
        'LightGBM': LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            importance_type='gain',
            random_state=random_state,
            verbose=-1,
            n_jobs=-1
        )
    }

    results = []

    for model_name, model in models.items():
        print(f"\nEvaluating {model_name}...")
        oof_probs = np.zeros(len(X))

        # 80%? ???? ??? ?????, ??? ?? ???? ?? ??? 20%? ???? ???? ?? ??? ?? oof_probs? ??
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
            X_tr = X.iloc[train_idx]
            X_va = X.iloc[val_idx]
            y_tr = y.iloc[train_idx]
            y_va = y.iloc[val_idx]

            model.fit(X_tr, y_tr)
            oof_probs[val_idx] = model.predict_proba(X_va)[:, 1]

        # PR curve ?? threshold ??
        # precision_recall_curve? ??? F2-Score? ??? ?? threshold ??
        precisions, recalls, thresholds = precision_recall_curve(y, oof_probs)

        # thresholds ??? precisions/recalls?? 1 ??
        f2_scores = (5 * precisions[:-1] * recalls[:-1]) / (
            4 * precisions[:-1] + recalls[:-1] + 1e-8
        )

        best_idx = np.argmax(f2_scores)
        best_thresh = thresholds[best_idx]
        best_f2 = f2_scores[best_idx]

        final_preds = (oof_probs >= best_thresh).astype(int)

        # ? ??? ??? ???
        results.append({
            'Model': model_name,
            'OOF F2': best_f2,
            'AP': average_precision_score(y, oof_probs),
            'ROC-AUC': roc_auc_score(y, oof_probs),
            'Precision': precision_score(y, final_preds, zero_division=0),
            'Recall': recall_score(y, final_preds, zero_division=0),
            'Threshold': best_thresh,
            'Pred_Positive_Rate': final_preds.mean()
        })

    results_df = pd.DataFrame(results).sort_values(by='OOF F2', ascending=False).reset_index(drop=True)
    return results_df

In [ ]:
# =========================
# ??
# =========================
comparison_df = evaluate_models_oof(X_train_sub_prep, y_train_sub)

print("\n=== Baseline OOF Comparison ===")
print(comparison_df)

* OOF-F2: Recall? 2? ???? ? ??
* AP: PR-Curve ?? ??? ??
* ROC-AUC
* Precision: ???. ????? ??? ?? ? ??? ??? ??? ? ????
* Recall: ???. ?? ??? ?? ? ??? ??????
* Threshold: ?? ??? ? ??? ? ????? ??? ???? ?? ??
* Pred_Positive_Rate: ?? ??? ? ??? ?%? ????? ??????

In [ ]:
import matplotlib.pyplot as plt

def plot_model_comparison(results_df):
    plt.figure(figsize=(8,5))
    plt.bar(results_df['Model'], results_df['OOF F2'])
    plt.title('Model Comparison (OOF F2 Score)')
    plt.xlabel('Model')
    plt.ylabel('F2 Score')
    plt.grid(axis='y')
    plt.show()

In [ ]:
plot_model_comparison(comparison_df)

## ?? ??

?? ???? ????, ? ??? ???? ?? ??? ?????? ??? ??? ???

In [ ]:
# =========================
# ??? ?? ??
# ??? ??? ??? ???? RF? XGBoost ??? ?? ???? ??? ?, ?? ??? ??? ?? ????? ??? ?? ???.
# =========================
def get_feature_importance(model_name, X, y, random_state=42):
    pos = y.sum()
    neg = len(y) - pos
    scale_pos_weight = neg / pos

    if model_name == 'RandomForest':
        model = RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            class_weight='balanced',
            n_jobs=-1,
            random_state=random_state
        )
    elif model_name == 'XGBoost':
        model = XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=4,
            min_child_weight=1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            scale_pos_weight=scale_pos_weight,
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=random_state,
            tree_method='hist',
            n_jobs=-1
        )
    else:
        raise ValueError("model_name must be 'RandomForest' or 'XGBoost'")

    model.fit(X, y)
    importances = pd.Series(model.feature_importances_, index=X.columns)
    importances = importances.sort_values(ascending=False)

    importance_df = pd.DataFrame({
        'Feature': importances.index,
        'Importance': importances.values
    })
    return importance_df


# =========================
# OOF ?? ??
# ??? ?? ??? ?? ???? ?? ??? ????? ??? ?????.
# ??? ?? ?? ??? F2-score? ?? ?? ??? ??? ????.
# =========================
def evaluate_single_model_oof(model_name, X, y, random_state=42):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

    pos = y.sum()
    neg = len(y) - pos
    scale_pos_weight = neg / pos

    if model_name == 'RandomForest':
        model = RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            class_weight='balanced',
            n_jobs=-1,
            random_state=random_state
        )
    elif model_name == 'XGBoost':
        model = XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=4,
            min_child_weight=1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            scale_pos_weight=scale_pos_weight,
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=random_state,
            tree_method='hist',
            n_jobs=-1
        )
    else:
        raise ValueError("model_name must be 'RandomForest' or 'XGBoost'")

    oof_probs = np.zeros(len(X))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        X_tr = X.iloc[train_idx]
        X_va = X.iloc[val_idx]
        y_tr = y.iloc[train_idx]

        model.fit(X_tr, y_tr)
        oof_probs[val_idx] = model.predict_proba(X_va)[:, 1]

    precisions, recalls, thresholds = precision_recall_curve(y, oof_probs)

    f2_scores = (5 * precisions[:-1] * recalls[:-1]) / (
        4 * precisions[:-1] + recalls[:-1] + 1e-8
    )

    best_idx = np.argmax(f2_scores)
    best_thresh = thresholds[best_idx]
    best_f2 = f2_scores[best_idx]

    final_preds = (oof_probs >= best_thresh).astype(int)

    result = {
        'Model': model_name,
        'n_features': X.shape[1],
        'OOF F2': best_f2,
        'AP': average_precision_score(y, oof_probs),
        'ROC-AUC': roc_auc_score(y, oof_probs),
        'Precision': precision_score(y, final_preds, zero_division=0),
        'Recall': recall_score(y, final_preds, zero_division=0),
        'Threshold': best_thresh,
        'Pred_Positive_Rate': final_preds.mean()
    }

    return result, oof_probs


# =========================
# top-k ?? ??
# ??? ?? ?? ??? ???? 10?, 20?,.. ??? 10?? ????? OOF F2 ??? ????.
# =========================
def compare_topk_features(model_name, X, y, topk_list=None, random_state=42):
    if topk_list is None:
        topk_list = [10, 20, 30, 40, 50, X.shape[1]]

    # ?? ?? ??? ? ? ??
    topk_list = [k for k in topk_list if k <= X.shape[1]]

    importance_df = get_feature_importance(model_name, X, y, random_state=random_state)

    results = []

    for k in topk_list:
        selected_features = importance_df['Feature'].head(k).tolist()
        X_k = X[selected_features].copy()

        print(f"Evaluating {model_name} with top-{k} features...")
        result, _ = evaluate_single_model_oof(model_name, X_k, y, random_state=random_state)
        results.append(result)

    results_df = pd.DataFrame(results).sort_values(by='OOF F2', ascending=False).reset_index(drop=True)
    return results_df, importance_df

In [ ]:
# =========================
# RF / XGB ?? ?? ??? ??
# =========================
topk_candidates = [10, 20, 30, 40, 50, X_train_sub_prep.shape[1]]

rf_topk_results, rf_importance_df = compare_topk_features(
    model_name='RandomForest',
    X=X_train_sub_prep,
    y=y_train_sub,
    topk_list=topk_candidates,
    random_state=42
)

xgb_topk_results, xgb_importance_df = compare_topk_features(
    model_name='XGBoost',
    X=X_train_sub_prep,
    y=y_train_sub,
    topk_list=topk_candidates,
    random_state=42
)

print("\n=== RandomForest Top-k Comparison ===")
print(rf_topk_results)

print("\n=== XGBoost Top-k Comparison ===")
print(xgb_topk_results)

print("\n=== RandomForest Top 20 Important Features ===")
print(rf_importance_df.head(20))

print("\n=== XGBoost Top 20 Important Features ===")
print(xgb_importance_df.head(20))

?? 40? ?? ???? ? ?? ?? ?? ??.

?? ???? ??????? ???

In [ ]:
# =========================
# top-40 feature ??
# ??? 40? ??? ???? ????.
# =========================
top_k = 40
selected_features = rf_importance_df['Feature'].head(top_k).tolist()

X_train_rf = X_train_sub_prep[selected_features].copy()


# =========================
# F2 scorer ??
# =========================
f2_scorer = make_scorer(fbeta_score, beta=2)


# =========================
# ?? ??
# =========================
rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)


# =========================
# ???? ?? ??
# =========================
param_dist = {
    'n_estimators': [300, 400, 500, 700],
    'max_depth': [None, 6, 8, 10, 12],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': ['sqrt', 'log2', 0.5, 0.8]
}


# =========================
# RandomizedSearch
# =========================
rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=25,
    scoring=f2_scorer,
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# train set?? F2 score ?? ?????
rf_search.fit(X_train_rf, y_train_sub)

print("\n=== Best RF Params ===")
print(rf_search.best_params_)

print("\n=== Best CV F2 ===")
print(rf_search.best_score_)

??? ?? ???????? ??? threshold ??

?? ?? ???? 5-Fold ?? ??? ???, ?? ???? ?? '?? ???'? ??? ?? ?? ? ? ??.

???? 5?? ?? ???? ????, ??? ??? ? ?? ??? ???? ?? ?? ??(oof_probs)? ?? ???.

In [ ]:
# best model ????
best_rf = rf_search.best_estimator_

# top-40 feature ??
top_k = 40
selected_features = rf_importance_df['Feature'].head(top_k).tolist()
X_train_rf = X_train_sub_prep[selected_features].copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_probs = np.zeros(len(X_train_rf))

for train_idx, val_idx in skf.split(X_train_rf, y_train_sub):
    X_tr = X_train_rf.iloc[train_idx]
    X_va = X_train_rf.iloc[val_idx]
    y_tr = y_train_sub.iloc[train_idx]

    best_rf.fit(X_tr, y_tr)
    oof_probs[val_idx] = best_rf.predict_proba(X_va)[:, 1]


# threshold ??
precisions, recalls, thresholds = precision_recall_curve(y_train_sub, oof_probs)

f2_scores = (5 * precisions[:-1] * recalls[:-1]) / (
    4 * precisions[:-1] + recalls[:-1] + 1e-8
)

best_idx = np.argmax(f2_scores)
best_thresh = thresholds[best_idx]
best_f2 = f2_scores[best_idx]

final_preds = (oof_probs >= best_thresh).astype(int)

print("\n=== Tuned RF OOF Results ===")
print("F2:", best_f2)
print("Threshold:", best_thresh)
print("Precision:", precision_score(y_train_sub, final_preds))
print("Recall:", recall_score(y_train_sub, final_preds))
print("Pred Positive Rate:", final_preds.mean())

In [ ]:
print("\n=== Threshold Sweep ===")

for t in np.linspace(0.1, 0.6, 10):
    preds = (oof_probs >= t).astype(int)
    
    f2 = fbeta_score(y_train_sub, preds, beta=2)
    prec = precision_score(y_train_sub, preds, zero_division=0)
    rec = recall_score(y_train_sub, preds, zero_division=0)
    rate = preds.mean()
    
    print(f"t={t:.2f} | F2={f2:.4f} | P={prec:.3f} | R={rec:.3f} | PosRate={rate:.3f}")

??? ??? ?????? ??? ??

t_low (???): 0.30
t_mid (???): 0.38
t_high (???): 0.45

*** 

## ?? ??? ??

?????? ?? ?? ?????, holdout data? ?? ????

In [ ]:
# top-40 feature
selected_features = rf_importance_df['Feature'].head(40).tolist()

X_train_rf = X_train_sub_prep[selected_features].copy()
X_holdout_rf = X_valid_holdout_prep[selected_features].copy()

best_rf = rf_search.best_estimator_
best_rf.fit(X_train_rf, y_train_sub)

holdout_probs = best_rf.predict_proba(X_holdout_rf)[:, 1]

??? ??? ??? ???? ?? ??? ?? ??? ??????? ??? ??

In [ ]:
# ?? threshold ??
threshold_list = [0.30, 0.32, 0.35, 0.38, 0.40, 0.43, 0.45, 0.49, 0.50, 0.55, 0.60, 0.65]

results = []

for t in threshold_list:
    preds = (holdout_probs >= t).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_valid_holdout, preds).ravel()

    results.append({
        'threshold': t,
        'F2': fbeta_score(y_valid_holdout, preds, beta=2, zero_division=0),
        'precision': precision_score(y_valid_holdout, preds, zero_division=0),
        'recall': recall_score(y_valid_holdout, preds, zero_division=0),
        'pred_positive_rate': preds.mean(),
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn
    })

threshold_results = pd.DataFrame(results).sort_values(
    by=['F2', 'precision'], ascending=[False, False]
).reset_index(drop=True)

print(threshold_results)

0.4? ??? ??!???

?? ?????? ??? 0.49? ?? ?? ??? ??

In [ ]:
# =========================
# ?? ??
# =========================
FINAL_THRESHOLD = 0.49
TOP_K = 40

# holdout ?? ?? ?? top-40 ?? ??
selected_features = rf_importance_df['Feature'].head(TOP_K).tolist()

# ?? flag ????? ???
flag_cols = [
    'Liability-Assets Flag',
    'Net Income Flag' 
]

# =========================
# train / test ?? ??
# =========================
X_full_train = df_train.drop(columns=['ID', 'Bankrupt?']).copy()
y_full_train = df_train['Bankrupt?'].copy()

X_test = df_test.drop(columns=['ID']).copy()
test_ids = df_test['ID'].copy()

print("Full train shape:", X_full_train.shape)
print("Test shape:", X_test.shape)
print("Positive rate in full train:", y_full_train.mean())

# =========================
# ???: full train? fit, test? transform
# =========================
final_preprocessor = FinancePreprocessor(
    flag_cols=flag_cols,
    clip_lower=0.01,
    clip_upper=0.99,
    skew_threshold=1.0
)

X_full_train_prep = final_preprocessor.fit_transform(X_full_train)
X_test_prep = final_preprocessor.transform(X_test)

print("Preprocessed full train shape:", X_full_train_prep.shape)
print("Preprocessed test shape:", X_test_prep.shape)
print("Number of log-transformed columns:", len(final_preprocessor.skewed_cols_))

# =========================
# ??? feature? ??
# =========================
X_full_train_final = X_full_train_prep[selected_features].copy()
X_test_final = X_test_prep[selected_features].copy()

print("Final train feature shape:", X_full_train_final.shape)
print("Final test feature shape:", X_test_final.shape)

# =========================
# ?? RF ?? ??
# (?? ?? ??)
# =========================
final_rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=6,
    min_samples_split=20,
    min_samples_leaf=1,
    max_features='log2',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# =========================
# ?? train?? ???
# =========================
final_rf.fit(X_full_train_final, y_full_train)

# =========================
# test ?? ?? + threshold ??
# =========================
test_probs = final_rf.predict_proba(X_test_final)[:, 1]
test_preds = (test_probs >= FINAL_THRESHOLD).astype(int)

print("Test predicted positive rate:", test_preds.mean())

# =========================
# ?? ?? ??
#    ???? ?? ?? ??? ?? ?? ??
# =========================
submission = pd.DataFrame({
    'ID': test_ids,
    'Bankrupt?': test_preds
})

***

## ??? ?? ??

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
submission_path = '/content/drive/MyDrive/result.csv'

submission.to_csv(submission_path, index=False)

print(f"Saved to Google Drive: {submission_path}")

***